# The action API — how Darab pulls my data

**Darab, this notebook is for you.** My side turns raw reality into decisions; you pull
those decisions from one endpoint:

```
GET http://localhost:8788/latest_action
```

It's your frame API, mirrored: I poll `:8787` for your sensor data; you poll `:8788` for
my actions. Same rules — poll, and dedupe on `action_id`.

In this demo, a **wav file stands in for a real voice near the robot** — someone saying
"Hey Jarvis, water the ficus". Everything else is the real thing: the real service
(`src/audio_on_demand.py`), the real Gemma, the real endpoint. Full manual:
`docs/v2_architecture/action_api.md`.

## How to run

Kernel: **gemma-lab**. No GUI needed. Run top-to-bottom; the Gemma load makes the first
action take ~1 min to appear.

In [1]:
import sys, os, time, json, subprocess, urllib.request, urllib.error

REPO = "/Users/asaphbrosh/Documents/Projects/Plantaf"
REC  = os.path.join(REPO, "experiments/audio_on_demand/recordings")
PORT = 8788

SCENARIOS = [                                   # (file, what is said, what we expect)
    ("tts_water_ficus.wav", "Hey Jarvis! Please water the ficus.",      "water_plant"),
    ("tts_question.wav",    "Hey Jarvis! How are my plants doing today?", "speak (an answer)"),
    ("tts_impossible.wav",  "Hey Jarvis! Please dance for me.",          "speak (a refusal)"),
]
for f, said, exp in SCENARIOS:
    print(f"{f:24}  “{said}”  →  expect {exp}")

tts_water_ficus.wav       “Hey Jarvis! Please water the ficus.”  →  expect water_plant
tts_question.wav          “Hey Jarvis! How are my plants doing today?”  →  expect speak (an answer)
tts_impossible.wav        “Hey Jarvis! Please dance for me.”  →  expect speak (a refusal)


## 1. The "voice"

Three spoken requests — pretend they happen near the robot. They ship as synthetic-voice
wavs and get joined into one continuous stream, as if a person spoke three times with
pauses. *(Asaf-only aside: `RECORD_MINE = True` re-records them with a real voice —
Darab, just run the cell.)*

In [2]:
import numpy as np, soundfile as sf

RECORD_MINE = False                     # flip to True, run, speak each line
if RECORD_MINE:
    import sounddevice as sd
    for f, said, _ in SCENARIOS:
        input(f"ENTER, then say: “{said}”  (6 s)")
        audio = sd.rec(int(6*16000), samplerate=16000, channels=1, dtype="int16")
        sd.wait()
        sf.write(os.path.join(REC, f), audio[:, 0], 16000)
        print("saved", f)

sil = np.zeros(16000, dtype=np.int16)   # 1 s gap
parts = []
for f, _, _ in SCENARIOS:
    a, sr = sf.read(os.path.join(REC, f), dtype="int16")
    parts += [a if a.ndim == 1 else a[:, 0], sil]
combined = os.path.join(REC, "tts_combined.wav")
sf.write(combined, np.concatenate(parts), 16000)
print(f"combined stream: {len(np.concatenate(parts))/16000:.1f} s → {os.path.basename(combined)}")

combined stream: 15.6 s → tts_combined.wav


## 2. My side — the producer

The real service, the same file that runs on the robot. It sleeps until the wake word,
records the sentence, asks Gemma, and publishes the decision on `:8788`. On the robot
you'd drop `--wav` and it listens to the live mic instead — nothing else changes:

```bash
python src/audio_on_demand.py --wav <recording> --brain gemma
```

In [3]:
LOG = os.path.join(REPO, "experiments/audio_on_demand/service.log")
logf = open(LOG, "w")
svc = subprocess.Popen(
    [sys.executable, os.path.join(REPO, "src/audio_on_demand.py"),
     "--wav", combined, "--brain", "gemma", "--port", str(PORT)],
    stdout=logf, stderr=subprocess.STDOUT, cwd=REPO)
print("service launched, pid", svc.pid, "— loading Gemma (~1 min), then eating the stream")

service launched, pid 68510 — loading Gemma (~1 min), then eating the stream


## 3. Your side — the consumer. **This cell is the whole point.**

This loop is everything you need on the robot: poll `latest_action`, and when
`action_id` is one you haven't seen, execute it — once. It's your own `api_demo.py`
pattern, pointed at my port. Below, "execute" just means print.

In [4]:
seen, actions, t0 = set(), [], time.time()
while len(actions) < len(SCENARIOS) and time.time() - t0 < 420:
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/latest_action", timeout=2) as r:
            a = json.loads(r.read())
        if a["action_id"] not in seen:            # dedupe — execute each id once
            seen.add(a["action_id"])
            actions.append(a)
            print(f"[{time.time()-t0:5.1f}s]  #{a['action_id']}  {a['action']}({a['args']})")
    except (urllib.error.URLError, urllib.error.HTTPError, OSError):
        pass                                       # 503 / not up yet — keep polling
    time.sleep(0.2)

svc.terminate(); logf.close()
print(f"\ncollected {len(actions)}/{len(SCENARIOS)} actions")

[ 50.7s]  #1  water_plant({'id': 'ficus', 'ml': 100})


[ 52.8s]  #2  daily_summary({})


[ 58.3s]  #3  speak({'text': "Sorry, I can't do that. I can approach a plant, scan the room, inspect a plant, water a plant, flag an issue, give a daily summary, or wait until a specific time."})



collected 3/3 actions


## 4. Under the hood of my side

The service log: each wake, each utterance, and the per-call brain time — call #1 pays
the one-time model load (~1 min); #2 and #3 show the warm cost (~4 s).

In [5]:
for line in open(LOG):
    if any(k in line for k in ("[ears]", "[action]", "[brain]", "[server]")):
        print(line.rstrip())

[server] GET http://localhost:8788/latest_action
[ears] asleep — waiting for the wake word (wav mode)
[ears] wake word heard — recording …
[ears] utterance closed (2.9 s) → brain
[brain] loading google/gemma-4-E4B-it on mps …
[action] #1  water_plant({'id': 'ficus', 'ml': 100})  [49.1 s]
[ears] wake word heard — recording …
[ears] utterance closed (3.4 s) → brain
[action] #2  daily_summary({})  [1.9 s]
[ears] wake word heard — recording …
[ears] utterance closed (2.8 s) → brain
[action] #3  speak({'text': "Sorry, I can't do that. I can approach a plant, scan the room, inspect a plant, water a plant, flag an issue, give a daily summary, or wait until a specific time."})  [5.6 s]
[ears] wav finished — still serving; Ctrl+C to quit


## Take-away for Darab

1. **One endpoint:** `GET :8788/latest_action`. Poll it, dedupe on `action_id`, execute
   each id once. The loop in section 3 is the whole integration.
2. **Payload:** `{action_id, ts, action, args, sim_mode}` — actions and args are always
   validated against the schema in `docs/v2_architecture/action_api.md`; you'll never
   see a malformed one.
3. **The wav was a stand-in.** On the robot the same service runs with no `--wav`,
   listening to the live mic through your `:8787` — already tested end-to-end with a
   real voice recording made by your GUI (`recordings/session_20260727_234304/`).
4. Known gaps on my side: args aren't grounded yet (`ml: 500` is invented until the
   `.md` knowledge layer exists); wake phrase is "Hey Jarvis" until we train "Hi
   Robot"; and whether Gemma E4B fits the Jetson's 8 GB is an open question for you.